In [138]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 데이터 불러오기

In [139]:
# customers = pd.read_csv('/content/drive/MyDrive/why-they-leave/retail-clickstream-analysis/raw/customers.csv')
products = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\dataset\products.csv")
events = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\dataset\events.csv")
sessions = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\dataset\sessions.csv")
orders = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\dataset\orders.csv")
order_items = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\dataset\order_items.csv")


# 보완재

In [140]:
import itertools
from collections import defaultdict

In [141]:
sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   session_id   120000 non-null  int64 
 1   customer_id  120000 non-null  int64 
 2   start_time   120000 non-null  object
 3   device       120000 non-null  object
 4   source       120000 non-null  object
 5   country      120000 non-null  object
dtypes: int64(2), object(4)
memory usage: 5.5+ MB


In [142]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 760958 entries, 0 to 760957
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   event_id      760958 non-null  int64  
 1   session_id    760958 non-null  int64  
 2   timestamp     760958 non-null  object 
 3   event_type    760958 non-null  object 
 4   product_id    682469 non-null  float64
 5   qty           143126 non-null  float64
 6   cart_size     44909 non-null   float64
 7   payment       33580 non-null   object 
 8   discount_pct  33580 non-null   float64
 9   amount_usd    33580 non-null   float64
dtypes: float64(5), int64(2), object(3)
memory usage: 58.1+ MB


In [143]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33580 entries, 0 to 33579
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        33580 non-null  int64  
 1   customer_id     33580 non-null  int64  
 2   order_time      33580 non-null  object 
 3   payment_method  33580 non-null  object 
 4   discount_pct    33580 non-null  int64  
 5   subtotal_usd    33580 non-null  float64
 6   total_usd       33580 non-null  float64
 7   country         33580 non-null  object 
 8   device          33580 non-null  object 
 9   source          33580 non-null  object 
dtypes: float64(2), int64(3), object(5)
memory usage: 2.6+ MB


In [144]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59163 entries, 0 to 59162
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        59163 non-null  int64  
 1   product_id      59163 non-null  int64  
 2   unit_price_usd  59163 non-null  float64
 3   quantity        59163 non-null  int64  
 4   line_total_usd  59163 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 2.3 MB


## 데이터 전처리
1. 시간 관련된 변수들 전부 datetime으로 변경
2. sessions와 events 테이블 session_id로 join <- session_id에 있는 customer_id, 즉 어떤 유저에 대한 행동인지를 알기 위해
3. orders와 order_items 테이블 order_id로 join <- order_itmems table에 있는 product_id, 즉 어떤 품목을 구매한건지 알기 위해
4. 2번 3번 join 해서 events table의 purchase null 값 채우기

4번까지 하면 한 유저에 대한 전체 아이템 - 행동 로그 데이터 셋 만들어짐.


In [145]:
# 1) events 테이블
events['timestamp'] = pd.to_datetime(events['timestamp'])

# 2) sessions 테이블
sessions['start_time'] = pd.to_datetime(sessions['start_time'])

# 3) orders 테이블
orders['order_time'] = pd.to_datetime(orders['order_time'])

In [146]:
# ==========================================
# 1. 기본 베이스: 세션과 유저 ID 결합
# ==========================================
# 모든 분석의 기본이 되는 이벤트+고객 기본 테이블 생성
df_base_events = pd.merge(
    events[['session_id', 'event_id', 'timestamp', 'event_type', 'product_id']], 
    sessions[['session_id', 'customer_id']], 
    on='session_id', 
    how='left'
)
df_base_events

,session_id,event_id,timestamp,event_type,product_id,customer_id
0,1,1,2021-12-27 00:08:36,page_view,93.0,12360
1,1,2,2021-12-27 00:16:36,page_view,1005.0,12360
2,1,3,2021-12-27 00:18:01,add_to_cart,1005.0,12360
3,1,4,2021-12-27 00:45:36,page_view,918.0,12360
4,1,5,2021-12-27 01:03:36,page_view,946.0,12360
...,...,...,...,...,...,...
760953,119999,760954,2025-04-22 10:44:56,checkout,NaN,19550
760954,119999,760955,2025-04-22 10:45:56,purchase,NaN,19550
760955,120000,760956,2023-08-24 15:18:53,page_view,762.0,453
760956,120000,760957,2023-08-24 15:34:53,page_view,772.0,453


In [147]:
# ==========================================
# 2. 비-구매 이벤트 분리 (add_to_cart, page_view 등)
# ==========================================
# 이미 product_id가 이벤트 로그에 바로 찍혀있는 녀석들만 추출
df_non_purchase = df_base_events[df_base_events['event_type'].isin(['add_to_cart', 'page_view'])].copy()
df_non_purchase

,session_id,event_id,timestamp,event_type,product_id,customer_id
0,1,1,2021-12-27 00:08:36,page_view,93.0,12360
1,1,2,2021-12-27 00:16:36,page_view,1005.0,12360
2,1,3,2021-12-27 00:18:01,add_to_cart,1005.0,12360
3,1,4,2021-12-27 00:45:36,page_view,918.0,12360
4,1,5,2021-12-27 01:03:36,page_view,946.0,12360
...,...,...,...,...,...,...
760951,119999,760952,2025-04-22 10:40:56,page_view,328.0,19550
760952,119999,760953,2025-04-22 10:43:56,page_view,244.0,19550
760955,120000,760956,2023-08-24 15:18:53,page_view,762.0,453
760956,120000,760957,2023-08-24 15:34:53,page_view,772.0,453


In [148]:
# ==========================================
# 3. 구매(purchase) 이벤트 전처리 (order_items와 조인하여 product_id 채우기)
# ==========================================
# 1) purchase 로그만 필터링
df_purchase_logs = df_base_events[df_base_events['event_type'] == 'purchase'].copy()

# 2) orders와 시간/고객 기준으로 inner_join해서 order_id 확보
df_purchase_with_order_id = pd.merge(
    df_purchase_logs,
    orders[['order_id', 'customer_id', 'order_time']],
    left_on=['customer_id', 'timestamp'],
    right_on=['customer_id', 'order_time'],
    how='inner'
)

# 3) order_items와 조인하여 결제된 진짜 product_id 컬럼 채우기
df_purchase_with_items = pd.merge(
    df_purchase_with_order_id,
    order_items[['order_id', 'product_id']],
    on='order_id',
    how='left'
)

# 4) 불필요해진 order_id, order_time 정리하고 product_id 컬럼명을 통일
# (left_join하면서 생긴 product_id_x(기존 결측치) 대신 product_id_y(진짜 상품ID) 사용)
df_purchase_final = df_purchase_with_items.drop(columns=['product_id_x', 'order_id', 'order_time'])
df_purchase_final = df_purchase_final.rename(columns={'product_id_y': 'product_id'})

# ==========================================
# 4. 두 데이터프레임 수직 결합 (Concat) 및 최종 정제
# ==========================================
# 컬럼 순서를 완전히 일치시킨 후 아래로 이어붙임.
common_columns = ['customer_id', 'session_id', 'event_id', 'timestamp', 'event_type', 'product_id']

df_integrated_logs = pd.concat([
    df_non_purchase[common_columns],
    df_purchase_final[common_columns]
], axis=0, ignore_index=True)

df_integrated_logs['product_id'] = df_integrated_logs['product_id'].astype(int)

# 시간순으로 정렬해서 유저 여정 흐름 맞추기
df_integrated_logs = df_integrated_logs.sort_values(by=['customer_id', 'timestamp']).reset_index(drop=True)

In [149]:
df_integrated_logs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 741632 entries, 0 to 741631
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   customer_id  741632 non-null  int64         
 1   session_id   741632 non-null  int64         
 2   event_id     741632 non-null  int64         
 3   timestamp    741632 non-null  datetime64[ns]
 4   event_type   741632 non-null  object        
 5   product_id   741632 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(1)
memory usage: 33.9+ MB


In [150]:
len(order_items) + len(df_non_purchase)

741632

In [151]:
df_integrated_logs

,customer_id,session_id,event_id,timestamp,event_type,product_id
0,1,95761,606842,2022-03-18 02:18:29,page_view,358
1,1,95761,606843,2022-03-18 02:47:29,page_view,468
2,1,95761,606844,2022-03-18 02:47:41,add_to_cart,468
3,1,95761,606845,2022-03-18 02:49:29,page_view,451
4,1,95761,606846,2022-03-18 03:02:29,page_view,428
...,...,...,...,...,...,...
741627,20000,34151,214812,2023-05-12 00:23:28,page_view,1145
741628,20000,34151,214813,2023-05-12 00:45:28,page_view,462
741629,20000,34151,214814,2023-05-12 00:47:28,page_view,928
741630,20000,34151,214815,2023-05-12 01:11:28,page_view,300


## 모델링
1. 가중치 정의, purchase: 0.6, add_to_cart: 0.3, page_view: 0.1

In [152]:
from sklearn.model_selection import train_test_split

# 1) 먼저 최소 2개 이상의 상품을 가진 '유효한 세션 ID' 리스트만 발라냅니다.
session_product_counts = df_integrated_logs.groupby('session_id')['product_id'].nunique()
valid_session_ids = session_product_counts[session_product_counts >= 2].index.tolist()

# 2) 이 '세션 ID' 리스트를 80% 학습용, 20% 평가용으로 분리합니다.
train_session_ids, test_session_ids = train_test_split(
    valid_session_ids, 
    test_size=0.2, 
    random_state=42
)

# 3) 원본 통합 로그(df_integrated_logs)에서 해당 세션 ID에 속하는 행들만 필터링하여 복사합니다.
# 이렇게 하면 customer_id, event_type, timestamp가 모두 살아있는 원본 형태가 유지됩니다!
train_df = df_integrated_logs[df_integrated_logs['session_id'].isin(train_session_ids)].copy()
test_df = df_integrated_logs[df_integrated_logs['session_id'].isin(test_session_ids)].copy()

In [153]:
train_df

,customer_id,session_id,event_id,timestamp,event_type,product_id
0,1,95761,606842,2022-03-18 02:18:29,page_view,358
1,1,95761,606843,2022-03-18 02:47:29,page_view,468
2,1,95761,606844,2022-03-18 02:47:41,add_to_cart,468
3,1,95761,606845,2022-03-18 02:49:29,page_view,451
4,1,95761,606846,2022-03-18 03:02:29,page_view,428
...,...,...,...,...,...,...
741627,20000,34151,214812,2023-05-12 00:23:28,page_view,1145
741628,20000,34151,214813,2023-05-12 00:45:28,page_view,462
741629,20000,34151,214814,2023-05-12 00:47:28,page_view,928
741630,20000,34151,214815,2023-05-12 01:11:28,page_view,300


In [154]:
from itertools import permutations
from collections import Counter

# ==========================================
# 1. 유저 행동별 가중치(Weight) 정의
# ==========================================
# 구매와 장바구니는 강한 보완 의도, page_view(같이 본 상품)는 모수 보충용
action_weights = {
    'purchase': 0.6,
    'add_to_cart': 0.3,
    'page_view': 0.1
}

# ==========================================
# 2. 세션별 가중치 기반 단독 빈도 & 동시 출현 빈도 집계
# ==========================================
# 학습 데이터셋(train_df)'만 사용

product_scores = Counter()  # 상품별 누적 가중치 점수 (분모)
pair_scores = Counter()     # 상품 쌍별 누적 가중치 점수 (분자)

# 세션 단위로 그룹바이하여 순회
for session_id, g_df in train_df.groupby('session_id'):
    # 1) 세션 내 각 상품이 가질 수 있는 행동별 가중치 합하기
    sess_prod_weights = g_df.groupby('product_id')['event_type'].apply(lambda x: sum(action_weights.get(et, 0.1) for et in x)).to_dict()
    unique_products = list(sess_prod_weights.keys())
    
    # 세션 내에 상품이 최소 2개 이상 존재해야 연관 규칙 조합 생성 가능
    if len(unique_products) < 2:
        continue
        
    # 2) 단독 상품 누적 점수 계산 (분모용)
    for prod, w in sess_prod_weights.items():
        product_scores[prod] += w
        
    # 3) 상품 쌍(Permutations) 누적 점수 계산 (분자용)
    # 두 상품이 함께 등장했을 때, 두 상품의 가중치 중 최소값(혹은 평균)을 결합 점수로 산출
    for i in range(len(unique_products)):
        for j in range(len(unique_products)):
            if i == j: 
                continue  # 자기 자신과의 조합은 제외
                
            p1 = unique_products[i]  # 타겟 상품 A
            p2 = unique_products[j]  # 추천 후보 B
            
            pair_weight = min(sess_prod_weights[p1], sess_prod_weights[p2])
            pair_scores[(p1, p2)] += pair_weight

In [155]:
import pandas as pd

# ==========================================
# 3. 가중치 기반 조건부 확률 P(B|A) 계산
# ==========================================
# 딕셔너리 미리 생성 (카테고리 맵 및 상품명 맵)
product_category_map = products.set_index('product_id')['category'].to_dict()
product_name_map = products.set_index('product_id')['name'].to_dict()  # 상품명 맵 추가

score_list = []
for (prod_A, prod_B), count_AB in pair_scores.items():
    count_A = product_scores[prod_A]
    
    # 조건부 확률 공식: 분자(동시 점수) / 분모(A의 누적 가중치 점수)
    p_B_given_A = count_AB / count_A if count_A > 0 else 0
    
    score_list.append({
        'prod_A': prod_A,
        'prod_B': prod_B,
        'co_occurrence_score': count_AB,  # 행동 가중치가 총합된 누적 점수
        'score': p_B_given_A
    })

df_weighted_scores = pd.DataFrame(score_list)

# ==========================================
# 4. 카테고리 및 상품명 맵핑, 비즈니스 룰 적용
# ==========================================
# 제품별 카테고리 딕셔너리 맵핑
df_weighted_scores['cat_A'] = df_weighted_scores['prod_A'].map(product_category_map)
df_weighted_scores['cat_B'] = df_weighted_scores['prod_B'].map(product_category_map)

# 제품별 상품명 딕셔너리 맵핑 추가
df_weighted_scores['prod_A_name'] = df_weighted_scores['prod_A'].map(product_name_map)
df_weighted_scores['prod_B_name'] = df_weighted_scores['prod_B'].map(product_name_map)

# 룰 적용: 동일 카테고리 제외 (보완재 추천 목적)
df_recs = df_weighted_scores[df_weighted_scores['cat_A'] != df_weighted_scores['cat_B']].copy()

# ==========================================
# 5. 타겟 상품별 순위(Rank) 생성 및 정렬
# ==========================================
# 각 prod_A 그룹 내에서 score를 기준으로 내림차순 순위 매기기
df_recs['rank'] = (
    df_recs.groupby('prod_A')['score']
    .rank(ascending=False, method='min')
    .astype(int)
)

# 각 타겟 상품(prod_A)별 순위 순으로 정렬
df_recs = df_recs.sort_values(by=['prod_A', 'rank'], ascending=[True, True])

# ==========================================
# 6. Top 5 추출 및 컬럼 순서 정리
# ==========================================
# 상위 5개 추출
top_n_recs = df_recs.groupby('prod_A').head(5)

# 가독성을 위해 ID, 이름, 카테고리가 짝지어 보이도록 컬럼 순서 재배치
column_order = [
    'prod_A', 'prod_A_name', 'cat_A', 
    'rank', 
    'prod_B', 'prod_B_name', 'cat_B', 
    'co_occurrence_score', 'score'
]
top_n_recs = top_n_recs[column_order]

# 주피터 노트북에서 결과 확인
print(top_n_recs.head(10))

        prod_A            prod_A_name        cat_A  rank  prod_B  \
628932       1     SSD MediumBlue 149  Electronics     1    1105   
852896       1     SSD MediumBlue 149  Electronics     2     256   
267584       1     SSD MediumBlue 149  Electronics     3     831   
911474       1     SSD MediumBlue 149  Electronics     3     389   
911477       1     SSD MediumBlue 149  Electronics     3     890   
763739       2  Keyboard DeepPink 696  Electronics     1     231   
797925       2  Keyboard DeepPink 696  Electronics     1     502   
797924       2  Keyboard DeepPink 696  Electronics     3     335   
25871        2  Keyboard DeepPink 696  Electronics     4    1026   
725044       2  Keyboard DeepPink 696  Electronics     4     771   

                          prod_B_name           cat_B  co_occurrence_score  \
628932   Action Figure BlueViolet 644            Toys                  1.1   
852896                 Lamp Beige 624  Home & Kitchen                  0.5   
267584           

In [156]:
# 깔끔하게 필요한 열만 순서대로 남기기
final_columns = [
    'prod_A_name', 'prod_B_name',
    'score',
    'rank'
]
top_n_recs = top_n_recs[final_columns]

# 주피터 노트북에서 결과 확인
print(top_n_recs.head(10))

                  prod_A_name                    prod_B_name     score  rank
628932     SSD MediumBlue 149   Action Figure BlueViolet 644  0.065868     1
852896     SSD MediumBlue 149                 Lamp Beige 624  0.029940     2
267584     SSD MediumBlue 149             Hoodie HotPink 449  0.023952     3
911474     SSD MediumBlue 149        Moisturizer Thistle 259  0.023952     3
911477     SSD MediumBlue 149        Hardcover LightCyan 711  0.023952     3
763739  Keyboard DeepPink 696  Air Fryer MediumTurquoise 924  0.062147     1
797925  Keyboard DeepPink 696  Conditioner DarkTurquoise 563  0.062147     1
797924  Keyboard DeepPink 696               Blender Pink 290  0.056497     3
25871   Keyboard DeepPink 696    Hardcover PaleTurquoise 230  0.022599     4
725044  Keyboard DeepPink 696               Socks Orange 300  0.022599     4


In [157]:
top_n_recs

,prod_A_name,prod_B_name,score,rank
628932,SSD MediumBlue 149,Action Figure BlueViolet 644,0.065868,1
852896,SSD MediumBlue 149,Lamp Beige 624,0.029940,2
267584,SSD MediumBlue 149,Hoodie HotPink 449,0.023952,3
911474,SSD MediumBlue 149,Moisturizer Thistle 259,0.023952,3
911477,SSD MediumBlue 149,Hardcover LightCyan 711,0.023952,3
...,...,...,...,...
442968,Building Blocks WhiteSmoke 876,Moisturizer MediumAquaMarine 749,0.019905,1
57408,Building Blocks WhiteSmoke 876,Paperback DarkGray 479,0.015166,2
217900,Building Blocks WhiteSmoke 876,Dumbbell GreenYellow 959,0.013270,3
302910,Building Blocks WhiteSmoke 876,E-book YellowGreen 947,0.013270,3


In [158]:
top_n_recs.to_csv(r"C:\Users\USER\OneDrive\Desktop\dataset\bowanjae.csv", index=False)

In [159]:
import pandas as pd

# ==========================================
# 1. 고유 ID 집합(Set) 추출
# ==========================================
# Train과 Test에 등장하는 유저 ID 추출
train_customers = set(train_df['customer_id'].unique())
test_customers = set(test_df['customer_id'].unique())

# Train과 Test에 등장하는 상품 ID 추출
train_products = set(train_df['product_id'].unique())
test_products = set(test_df['product_id'].unique())

# ==========================================
# 2. 유저(Customer) 불일치 검증
# ==========================================
overlap_customers = train_customers.intersection(test_customers)
only_test_customers = test_customers - train_customers

customer_overlap_ratio = len(overlap_customers) / len(test_customers) * 100 if test_customers else 0
customer_cold_start_ratio = len(only_test_customers) / len(test_customers) * 100 if test_customers else 0

# ==========================================
# 3. 상품(Product) 불일치 검증 (추천 공백의 직접적 원인)
# ==========================================
overlap_products = train_products.intersection(test_products)
only_test_products = test_products - train_products

product_overlap_ratio = len(overlap_products) / len(test_products) * 100 if test_products else 0
product_cold_start_ratio = len(only_test_products) / len(test_products) * 100 if test_products else 0

# ==========================================
# 4. 결과 출력
# ==========================================
print("====== 📊 [가설 검증] Train vs Test 데이터 불일치 리포트 ======\n")
print(f"1. 유저(Customer) 측면:")
print(f"   - Test셋 전체 유저 수: {len(test_customers)}명")
print(f"   - Train셋과 '겹치는' 유저 비율: {customer_overlap_ratio:.2f}% (이 유저들은 취향 학습됨)")
print(f"   - Train셋에 '없는' 신규 유저 비율: {customer_cold_start_ratio:.2f}% ⚠️ (유저 불일치 발생)")

print(f"\n2. 상품(Product) 측면:")
print(f"   - Test셋 전체 상품 수: {len(test_products)}개")
print(f"   - Train셋과 '겹치는' 상품 비율: {product_overlap_ratio:.2f}%")
print(f"   - Train셋에 '없는' 신규 상품 비율: {product_cold_start_ratio:.2f}% ⚠️ (추천 불가 상품)")

====== 📊 [가설 검증] Train vs Test 데이터 불일치 리포트 ======

1. 유저(Customer) 측면:
   - Test셋 전체 유저 수: 12954명
   - Train셋과 '겹치는' 유저 비율: 98.70% (이 유저들은 취향 학습됨)
   - Train셋에 '없는' 신규 유저 비율: 1.30% ⚠️ (유저 불일치 발생)

2. 상품(Product) 측면:
   - Test셋 전체 상품 수: 1197개
   - Train셋과 '겹치는' 상품 비율: 100.00%
   - Train셋에 '없는' 신규 상품 비율: 0.00% ⚠️ (추천 불가 상품)


In [161]:
# ==========================================
# 1) 추천 사전(Dictionary) 생성 및 ID 기준 통일
# ==========================================
# 매칭을 위해 top_n_recs가 아닌, ID와 이름이 모두 살아있던 df_recs(또는 top_n_recs의 원본 단계)에서 
# '상품 ID(prod_A)'를 key로, '추천 상품 ID(prod_B)'를 value 리스트로 그룹화합니다.
recs_dict = df_recs.groupby('prod_A')['prod_B'].apply(list).to_dict()

hit_count = 0
total_cases = 0

# ==========================================
# 2) 테스트 데이터를 세션 단위로 그룹화하여 리스트로 변환
# ==========================================
# test_df를 session_id 기준으로 묶어 한 세션에 등장한 product_id들을 리스트로 추출합니다.
test_sessions = test_df.groupby('session_id')['product_id'].apply(list).values

# 3) 테스트 세션을 순회하며 검증
for products in test_sessions:
    # 세션 내에 유니크한 상품이 최소 2개 이상일 때만 검증 진행
    unique_products = list(set(products))
    if len(unique_products) < 2:
        continue
        
    for i, target_prod in enumerate(unique_products):
        # 해당 타겟 상품 ID에 대한 추천 리스트가 없으면 패스
        if target_prod not in recs_dict:
            continue
            
        # 타겟 상품을 제외한 '실제 함께 구매/조회된 정답 보완재 ID 리스트'
        actual_complements = unique_products[:i] + unique_products[i+1:]
        
        # 알고리즘이 추천한 Top N 리스트 (앞에서 5개만 슬라이싱)
        predicted_recs = recs_dict[target_prod][:5]
        
        # 추천 리스트 중 하나라도 실제 정답 리스트에 있으면 HIT 1건으로 간주
        has_hit = any(prod in actual_complements for prod in predicted_recs)
        
        if has_hit:
            hit_count += 1
        total_cases += 1

# ==========================================
# 4) 최종 HIT_RATE 산출
# ==========================================
hit_rate = hit_count / total_cases if total_cases > 0 else 0
print(f"기본 알고리즘 HIT_RATE: {hit_rate:.4f}")

기본 알고리즘 HIT_RATE: 0.0175
